# Experiment H6: Learning-rate artifact check for final training loss

**Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/H6_LR_ARTIFACT_CHECK/run_experiment.py`

This notebook is a reporting and interpretation companion to the script in the same directory.
It deliberately answers a narrow question on a toy benchmark: on this 2-layer 4x4 deep-linear
regression problem, is the large final-loss gap between vanilla Muon at `lr=0.02` and
curvature-rescaled Muon at `lr=0.02` largely a learning-rate artifact?

**Truthful scope.**
- The benchmark reports training loss after a fixed 500-step optimization budget.
- It does **not** directly measure curvature, generalization, or a universal optimizer advantage.
- Because the best settings can occur on sweep boundaries, the notebook uses **best tested LR**
  language rather than claiming a fully localized optimum.

## Imports, notebook-safe path handling, and counterpart loading

The notebook does not re-implement the experiment loop. Instead, it resolves the repository
root in a notebook-safe way, imports the script module, and uses the script's structured
`run_experiment()` return value for all tables and figures below.

In [ ]:
from pathlib import Path
import importlib.util
import textwrap
import time

import numpy as np

try:
    from IPython import get_ipython
except Exception:
    get_ipython = None

import matplotlib
if get_ipython is None or get_ipython() is None:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

try:
    import pandas as pd
except Exception:
    pd = None

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


def find_repo_root(start):
    start = Path(start).resolve()
    experiment_rel = Path('experiments/Tier_1_Core_Mechanism_Tests/H6_LR_ARTIFACT_CHECK/run_experiment.py')
    for candidate in [start, *start.parents]:
        if (candidate / experiment_rel).exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root from the current working directory.')


REPO_ROOT = find_repo_root(Path.cwd())
EXPERIMENT_DIR = REPO_ROOT / 'experiments' / 'Tier_1_Core_Mechanism_Tests' / 'H6_LR_ARTIFACT_CHECK'
SCRIPT_PATH = EXPERIMENT_DIR / 'run_experiment.py'
NOTEBOOK_PATH = EXPERIMENT_DIR / 'run_experiment.ipynb'

spec = importlib.util.spec_from_file_location('h6_lr_artifact_check', SCRIPT_PATH)
h6 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h6)


def show_table(rows):
    if pd is not None:
        df = pd.DataFrame(rows)
        display(df)
        return df
    for row in rows:
        print(row)
    return rows


def summarize_condition(display_name, condition, win_count=None):
    summary = condition['summary']
    row = {
        'condition': display_name,
        'lr': condition['lr'],
        'mean_final_loss': summary['mean'],
        'median_final_loss': summary['median'],
        'std_final_loss': summary['std'],
        'finite_fraction': summary['finite_fraction'],
    }
    if win_count is not None:
        row['seed_win_count'] = win_count
    return row


def ecdf(values):
    values = np.asarray(values, dtype=float)
    x = np.sort(values)
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y


## Reproducibility, runtime, and configuration

The cell below executes the script's reusable `run_experiment()` function directly. This keeps
the notebook and CLI behavior aligned while exposing raw per-LR losses, seedwise key-condition
results, rescaling statistics, and retained loss histories for analysis.

In [ ]:
notebook_wall_start = time.perf_counter()
results = h6.run_experiment()
notebook_wall_seconds = time.perf_counter() - notebook_wall_start

reproducibility_rows = [
    {'item': 'Experiment ID', 'value': results['experiment_id']},
    {'item': 'Scope', 'value': results['scope']},
    {'item': 'Script path', 'value': results['paths']['script']},
    {'item': 'Notebook path', 'value': results['paths']['notebook']},
    {'item': 'CLI command', 'value': results['reproducibility']['script_command']},
    {'item': 'Script-reported runtime (s)', 'value': round(results['runtime_seconds'], 3)},
    {'item': 'Notebook wall time for run_experiment() (s)', 'value': round(notebook_wall_seconds, 3)},
    {'item': 'Seeds', 'value': results['seeds']},
    {'item': 'Init seeds', 'value': results['init_seeds']},
    {'item': 'Vanilla LR grid', 'value': results['config']['vanilla_lrs']},
    {'item': 'SGD LR grid', 'value': results['config']['sgd_lrs']},
    {'item': 'Expected min-clamp LR reference', 'value': results['selected_lrs']['expected_lr_from_min_clamp']},
]
show_table(reproducibility_rows)
print('Verdict category:', results['verdict']['category'])

## Sweep-level results: final loss vs tested learning rate

The script's primary sweep measures **final training loss after 500 steps** for vanilla Muon and
SGD across discrete LR grids. The figure and tables below therefore support claims only about
those tested grid points, not a continuous LR optimum.

In [ ]:
vanilla_rows = []
for lr in results['vanilla']['lr_grid']:
    summary = results['vanilla']['by_lr'][lr]['summary']
    vanilla_rows.append({
        'method': 'Vanilla Muon',
        'lr': lr,
        'mean_final_loss': summary['mean'],
        'median_final_loss': summary['median'],
        'std_final_loss': summary['std'],
        'finite_fraction': summary['finite_fraction'],
    })

sgd_rows = []
for lr in results['sgd']['lr_grid']:
    summary = results['sgd']['by_lr'][lr]['summary']
    sgd_rows.append({
        'method': 'SGD',
        'lr': lr,
        'mean_final_loss': summary['mean'],
        'median_final_loss': summary['median'],
        'std_final_loss': summary['std'],
        'finite_fraction': summary['finite_fraction'],
    })

print('Vanilla Muon LR sweep')
show_table(vanilla_rows)
print('SGD LR sweep')
show_table(sgd_rows)

vanilla_lrs = np.asarray(results['vanilla']['lr_grid'], dtype=float)
vanilla_means = np.asarray([results['vanilla']['by_lr'][lr]['summary']['mean'] for lr in vanilla_lrs])
sgd_lrs = np.asarray(results['sgd']['lr_grid'], dtype=float)
sgd_means = np.asarray([results['sgd']['by_lr'][lr]['summary']['mean'] for lr in sgd_lrs])

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(vanilla_lrs, vanilla_means, marker='o', linewidth=2, label='Vanilla Muon')
ax.plot(sgd_lrs, sgd_means, marker='s', linewidth=2, label='SGD')
ax.scatter(
    [results['rescaled']['original']['lr']],
    [results['rescaled']['original']['summary']['mean']],
    marker='D', s=80, label='Rescaled Muon @ 0.02'
)
ax.scatter(
    [results['rescaled']['best_vanilla_lr']['lr']],
    [results['rescaled']['best_vanilla_lr']['summary']['mean']],
    marker='^', s=90, label='Rescaled Muon @ vanilla best tested LR'
)
ax.axvline(results['selected_lrs']['expected_lr_from_min_clamp'], linestyle=':', color='black', alpha=0.6,
           label='0.02 × 0.1 reference')
ax.axvline(results['vanilla']['best_tested_lr'], linestyle='--', color='C0', alpha=0.5,
           label='Vanilla best tested LR')
ax.axvline(results['sgd']['best_tested_lr'], linestyle='--', color='C1', alpha=0.5,
           label='SGD best tested LR')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Learning rate')
ax.set_ylabel('Mean final training loss across seeds')
ax.set_title('Final-loss sweep on the H6 toy benchmark')
ax.legend(loc='best', fontsize=9)
ax.grid(True, which='both', alpha=0.25)
plt.tight_layout()
plt.show()

if results['vanilla']['best_on_lower_boundary'] or results['vanilla']['best_on_upper_boundary']:
    print('Boundary caveat: vanilla Muon attains its best tested LR on a sweep edge.')
if results['sgd']['best_on_lower_boundary'] or results['sgd']['best_on_upper_boundary']:
    print('Boundary caveat: SGD also attains its best tested LR on a sweep edge.')

**Interpretation.**

- The notebook and script agree on the same sweep results because they share the same returned
  `results` object.
- The default-LR question is about the gap between vanilla Muon and rescaled Muon at `lr=0.02`.
  The broader tuning question asks whether vanilla Muon already wins once its LR is tuned.
- A sweep-edge minimum means "best tested LR" should be read literally.

## Key-condition comparison

The next figure focuses on the four conditions most relevant to the pair-level question:

1. vanilla Muon at its best tested LR,
2. rescaled Muon at the original `lr=0.02`,
3. rescaled Muon at the vanilla best tested LR,
4. SGD at its best tested LR.

The left panel is a paired, seedwise comparison; the right panel summarizes the cross-seed
distribution on a log-loss axis.

In [ ]:
condition_order = results['comparisons']['paired_condition_order']
conditions = results['comparisons']['conditions']
win_counts = results['comparisons']['winner_counts']

key_rows = [
    summarize_condition(conditions[key]['display_name'], conditions[key], win_counts[key])
    for key in condition_order
]
show_table(key_rows)

labels = [conditions[key]['display_name'] for key in condition_order]
loss_matrix = np.vstack([conditions[key]['final_losses'] for key in condition_order]).T

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
x = np.arange(len(condition_order))

for seed_losses in loss_matrix:
    axes[0].plot(x, seed_losses, marker='o', alpha=0.35, color='gray')
axes[0].plot(x, np.median(loss_matrix, axis=0), marker='o', linewidth=2.5, color='black', label='Median across seeds')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=20, ha='right')
axes[0].set_yscale('log')
axes[0].set_ylabel('Final training loss')
axes[0].set_title('Paired seedwise comparison')
axes[0].grid(True, which='both', alpha=0.25)
axes[0].legend(loc='best')

box_data = [conditions[key]['final_losses'] for key in condition_order]
axes[1].boxplot(box_data, tick_labels=labels, showmeans=True)
axes[1].set_yscale('log')
axes[1].set_ylabel('Final training loss')
axes[1].set_title('Distribution across seeds')
axes[1].grid(True, which='both', alpha=0.25)
for tick in axes[1].get_xticklabels():
    tick.set_rotation(20)
    tick.set_ha('right')

plt.tight_layout()
plt.show()

print('Seed win counts across the paired comparison set:')
for key in condition_order:
    print(f"  {conditions[key]['display_name']}: {win_counts[key]}")

**Interpretation.**

- If vanilla Muon at the best tested LR already beats rescaled Muon at `lr=0.02`, then the
  original default-LR comparison is not enough to establish added value from rescaling.
- If rescaled Muon becomes worse when applied at the vanilla best tested LR, then the broader
  claim "rescaling helps even after tuning LR" is not supported by this sweep.

## Rescaling behavior: scale-factor distribution and clamp summary

The rescaling mechanism is only partially visible through final loss. This section therefore
surfaces the retained per-step mean scale factors for the two rescaled conditions.

In [ ]:
rescaled_rows = []
for key in ['original', 'best_vanilla_lr']:
    condition = results['rescaled'][key]
    summary = condition['scale_summary']
    rescaled_rows.append({
        'condition': condition['display_name'],
        'lr': condition['lr'],
        'mean_scale': summary['mean'],
        'median_scale': summary['median'],
        'std_scale': summary['std'],
        'min_scale': summary['min'],
        'max_scale': summary['max'],
        'min_clamp_fraction': summary['min_clamp_fraction'],
        'max_clamp_fraction': summary['max_clamp_fraction'],
    })
show_table(rescaled_rows)

orig_scales = results['rescaled']['original']['flattened_scales']
best_scales = results['rescaled']['best_vanilla_lr']['flattened_scales']
bins = np.linspace(
    min(orig_scales.min(), best_scales.min()),
    max(orig_scales.max(), best_scales.max()),
    50,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].hist(orig_scales, bins=bins, alpha=0.65, density=True, label='Rescaled @ 0.02')
axes[0].hist(best_scales, bins=bins, alpha=0.65, density=True,
             label=f"Rescaled @ vanilla best tested LR ({results['selected_lrs']['vanilla_best_tested']})")
axes[0].axvline(results['config']['scale_min'], linestyle='--', color='black', alpha=0.7, label='Min clamp')
axes[0].set_xlabel('Per-step mean scale factor')
axes[0].set_ylabel('Density')
axes[0].set_title('Histogram of retained scale factors')
axes[0].legend(loc='best', fontsize=9)
axes[0].grid(True, alpha=0.25)

x1, y1 = ecdf(orig_scales)
x2, y2 = ecdf(best_scales)
axes[1].plot(x1, y1, linewidth=2, label='Rescaled @ 0.02')
axes[1].plot(x2, y2, linewidth=2,
             label=f"Rescaled @ vanilla best tested LR ({results['selected_lrs']['vanilla_best_tested']})")
axes[1].axvline(results['config']['scale_min'], linestyle='--', color='black', alpha=0.7, label='Min clamp')
axes[1].set_xlabel('Per-step mean scale factor')
axes[1].set_ylabel('Empirical CDF')
axes[1].set_title('Cumulative view of scale-factor usage')
axes[1].grid(True, alpha=0.25)
axes[1].legend(loc='best', fontsize=9)

plt.tight_layout()
plt.show()

**Interpretation.**

- Heavy mass near the minimum clamp at `lr=0.02` supports the observation that the original
  rescaled run often behaves like a strong LR reduction.
- But this does **not** by itself prove global equivalence to a constant `0.1` multiplier,
  because the scale distribution can change when the base LR changes.

## Optimization trajectories

Unlike the original notebook, this first completion pass retains the loss histories already
computed inside the trainers. That makes it possible to compare median training trajectories,
not only final endpoints.

In [ ]:
trajectory_order = results['comparisons']['trajectory_condition_order']
trajectory_conditions = results['comparisons']['conditions']
steps = np.arange(results['config']['num_steps'] + 1)

fig, ax = plt.subplots(figsize=(10, 5.5))
for key in trajectory_order:
    histories = np.asarray(trajectory_conditions[key]['loss_histories'], dtype=float)
    median = np.median(histories, axis=0)
    q25 = np.percentile(histories, 25, axis=0)
    q75 = np.percentile(histories, 75, axis=0)
    ax.plot(steps, median, linewidth=2, label=trajectory_conditions[key]['display_name'])
    ax.fill_between(steps, q25, q75, alpha=0.12)

ax.set_yscale('log')
ax.set_xlabel('Training step')
ax.set_ylabel('Loss')
ax.set_title('Median loss trajectories with interquartile bands')
ax.grid(True, which='both', alpha=0.25)
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

**Trajectory caveat.**

These trajectories still live inside the same toy benchmark and training budget. They improve
interpretability, but they do not change the experiment's scope: the primary quantity being
compared remains training loss on this synthetic setup.

## Decision criteria T1/T2/T3

The original notebook talked about these checks as if they were formal tests. In this first
implementation pass they are reported more honestly as **operational decision criteria** on the
final-loss summaries.

In [ ]:
t1 = results['tests']['T1']
t2 = results['tests']['T2']
t3 = results['tests']['T3']

test_rows = [
    {
        'criterion': 'T1',
        'question': t1['question'],
        'definition': t1['definition'],
        'observed_metric': t1['ratio_best_to_expected'],
        'pass': t1['pass'],
        'note': 'Boundary-limited best tested LR' if (results['vanilla']['best_on_lower_boundary'] or results['vanilla']['best_on_upper_boundary']) else 'Interior tested point',
    },
    {
        'criterion': 'T2',
        'question': t2['question'],
        'definition': t2['definition'],
        'observed_metric': t2['vanilla_best_over_rescaled_original_ratio'],
        'pass': t2['pass'],
        'note': f"parity gap = {t2['parity_gap_percent']:.1f}%, log10 ratio = {t2['log10_vanilla_best_over_rescaled_original_ratio']:.2f}",
    },
    {
        'criterion': 'T3',
        'question': t3['question'],
        'definition': t3['definition'],
        'observed_metric': t3['vanilla_best_over_rescaled_best_ratio'],
        'pass': t3['pass'],
        'note': f"rescaled/vanilla = {t3['rescaled_best_over_vanilla_best_ratio']:.4e}, log10 ratio = {t3['log10_vanilla_best_over_rescaled_best_ratio']:.2f}",
    },
]
show_table(test_rows)

reference_rows = [
    {'reference_condition': 'Vanilla Muon @ 0.02', 'mean_final_loss': results['tests']['reference_values']['vanilla_original_mean']},
    {'reference_condition': f"Vanilla Muon @ best tested lr={results['selected_lrs']['vanilla_best_tested']}", 'mean_final_loss': results['tests']['reference_values']['vanilla_best_mean']},
    {'reference_condition': 'Rescaled Muon @ 0.02', 'mean_final_loss': results['tests']['reference_values']['rescaled_original_mean']},
    {'reference_condition': f"Rescaled Muon @ vanilla best tested lr={results['selected_lrs']['vanilla_best_tested']}", 'mean_final_loss': results['tests']['reference_values']['rescaled_best_mean']},
    {'reference_condition': f"SGD @ best tested lr={results['selected_lrs']['sgd_best_tested']}", 'mean_final_loss': results['sgd']['best_tested_summary']['mean']},
]
show_table(reference_rows)

**Interpretation.**

- **T1** asks whether the vanilla best tested LR sits near the naive `0.02 × 0.1` reference.
  It is only a coarse consistency check.
- **T2** asks whether tuned vanilla matches rescaled Muon at `lr=0.02`.
- **T3** asks the broader question: after tuning vanilla LR, does adding rescaling still help?

Reporting ratios and log-ratios is more stable than quoting giant percentage improvements when
one of the losses is already extremely small.

## Per-seed winners across the paired comparison set

The table below makes the paired comparison fully explicit at the seed level.

In [ ]:
per_seed_rows = []
for row in results['comparisons']['per_seed_winners']:
    loss_dict = row['losses']
    per_seed_rows.append({
        'seed': row['seed'],
        'init_seed': row['init_seed'],
        'vanilla_best_loss': loss_dict['vanilla_best'],
        'rescaled_0.02_loss': loss_dict['rescaled_original'],
        'rescaled_best_lr_loss': loss_dict['rescaled_best'],
        'sgd_best_loss': loss_dict['sgd_best'],
        'winner': row['winner_display_name'],
    })
show_table(per_seed_rows)

## Calibrated conclusion

The final cell produces a data-dependent conclusion that explicitly separates the narrow
default-LR-artifact question from the broader question of whether tuned vanilla already
outperforms rescaling.

In [ ]:
ratio_default_vs_rescaled = results['comparisons']['ratios']['vanilla_original_over_rescaled_original']
ratio_default_vs_vanilla_best = results['comparisons']['ratios']['vanilla_original_over_vanilla_best']
t3 = results['tests']['T3']
verdict = results['verdict']

conclusion_lines = [
    'Calibrated conclusion',
    '---------------------',
    f"Verdict category: {verdict['category']}",
    '',
    'Narrow default-LR-artifact question:',
    f"- Vanilla Muon at lr=0.02 is worse than rescaled Muon at lr=0.02 by {ratio_default_vs_rescaled:.1f}x in mean final loss.",
    f"- Vanilla Muon also improves by {ratio_default_vs_vanilla_best:.1f}x when moved from lr=0.02 to its best tested LR ({results['selected_lrs']['vanilla_best_tested']}).",
    '',
    'Broader tuning question:',
    f"- Vanilla(best tested) / Rescaled(0.02) = {results['tests']['T2']['vanilla_best_over_rescaled_original_ratio']:.4e} (log10 ratio = {results['tests']['T2']['log10_vanilla_best_over_rescaled_original_ratio']:.2f}).",
    f"- Vanilla(best tested) / Rescaled(best tested LR) = {t3['vanilla_best_over_rescaled_best_ratio']:.4e} (log10 ratio = {t3['log10_vanilla_best_over_rescaled_best_ratio']:.2f}).",
    '',
    f"Summary: {verdict['summary']}",
]

if verdict['boundary_notes']:
    conclusion_lines.append('')
    conclusion_lines.append('Boundary caveats:')
    conclusion_lines.extend([f"- {note}" for note in verdict['boundary_notes']])

conclusion_lines.extend([
    '',
    'Recommended next verification if the goal is a stronger scientific claim:',
    '- extend the vanilla LR sweep below 0.0005 so the best tested vanilla LR is no longer boundary-limited;',
    '- optionally extend the SGD sweep above 0.5 for the same reason;',
    '- then rerun the parity checks to see whether the current tuned-vanilla-vs-rescaled conclusion persists.',
])

print('\n'.join(conclusion_lines))
